# 05. 機能解析 (Functional Analysis)

DEG結果に対してGene Ontology (GO) エンリッチメント解析、パスウェイ解析、
Pre-ranked GSEA、および遺伝子ネットワーク可視化を行います。

**カーネル**: RNA-seq (Python) / rnaseq_env  
**入力**: `results/deg_*_vs_*.csv`  
**出力**: GO/パスウェイエンリッチメント結果 (CSV, HTML)

> **インターネット接続が必要です**: g:Profiler API を使用するため、実行時にインターネット接続が必要です。

### 使用ツール・ライセンス (全て商用利用可)

| ツール | ライセンス | 用途 |
|--------|-----------|------|
| gprofiler-official | BSD | GO/パスウェイエンリッチメント (g:Profiler API) |
| gseapy | BSD-3 | Pre-ranked GSEA |
| networkx | BSD-3 | エンリッチメントネットワーク構築 |
| plotly | MIT | インタラクティブ可視化 |

> **注意**: パスウェイDBは **Reactome** (CC BY 4.0) と **WikiPathways** (CC BY 3.0) を使用します。  
> KEGG は商用ライセンスが必要なため除外しています。

## 設定

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from gprofiler import GProfiler
import gseapy as gp
import glob, os, warnings
warnings.filterwarnings("ignore")

# === 設定 ===
LFC_THRESHOLD = 1.0
PADJ_THRESHOLD = 0.05
ORGANISM = "hsapiens"

# GOエンリッチメント解析パラメータ
GO_PVAL_THRESHOLD = 0.05      # エンリッチメントP値閾値
MAX_TERM_SIZE = 500            # 大きすぎるGO termを除外
MIN_TERM_SIZE = 5              # 小さすぎるGO termを除外
TOP_N_TERMS = 20               # 可視化する上位term数

# ネットワーク可視化パラメータ
JACCARD_THRESHOLD = 0.3        # term間のJaccard係数閾値

# GSEA パラメータ
GSEA_PERMUTATIONS = 1000       # GSEA permutation数
GSEA_MIN_SIZE = 15             # 最小遺伝子セットサイズ
GSEA_MAX_SIZE = 500            # 最大遺伝子セットサイズ

# 色の定義 (04_Visualizationと統一)
COLOR_UP = "#E74C3C"
COLOR_DOWN = "#3498DB"
COLOR_NS = "#BDC3C7"

# 出力ディレクトリ
OUTPUT_DIR = "results/functional"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("設定完了")

## ツール確認

In [ ]:
# 必要パッケージの確認
missing = []
for pkg, name in [("gprofiler", "gprofiler-official"), ("gseapy", "gseapy"), ("networkx", "networkx")]:
    try:
        __import__(pkg)
        print(f"  {name}: OK")
    except ImportError:
        print(f"  {name}: NOT FOUND")
        missing.append(name)

if missing:
    raise ImportError(
        f"以下のパッケージがインストールされていません: {', '.join(missing)}\n"
        f"conda activate rnaseq_env && pip install {' '.join(missing)}"
    )

## データ読み込みと遺伝子ID変換

DEG結果を読み込み、GENCODE遺伝子ID（ENSG...バージョン付き）からバージョンを除去し、
g:Profiler の `gconvert` でHGNCシンボルに変換します。

In [ ]:
# DEG結果ファイルの読み込み
result_files = sorted(glob.glob("results/deg_*_vs_*.csv"))

if not result_files:
    raise FileNotFoundError(
        "DEG結果ファイルが見つかりません: results/deg_*_vs_*.csv\n"
        "先に 03_DEG_Analysis.ipynb を実行してください。"
    )

all_results = {}
for f in result_files:
    df = pd.read_csv(f)
    if "comparison" not in df.columns:
        raise ValueError(f"'comparison' カラムが見つかりません: {f}")
    # GENCODE バージョン番号を除去 (ENSG00000141510.18 → ENSG00000141510)
    df["gene_id_stripped"] = df["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    comp_name = df["comparison"].iloc[0]
    all_results[comp_name] = df

print(f"比較条件数: {len(all_results)}")
for name, df in all_results.items():
    print(f"  {name}: {len(df)} genes")

In [ ]:
# gene_nameを優先的に使用（02で生成したgene_id_to_name.csvベース）
# g:Profilerはgene_nameがない場合のフォールバックとして使用
all_genes = set()
for df in all_results.values():
    all_genes.update(df["gene_id_stripped"].dropna().tolist())

print(f"変換対象遺伝子数: {len(all_genes)}")

# まずローカルのgene_id_to_name.csvから変換
if os.path.exists("results/gene_id_to_name.csv"):
    _gmap_df = pd.read_csv("results/gene_id_to_name.csv")
    id_map = dict(zip(_gmap_df["gene_id"], _gmap_df["gene_name"]))
    print(f"ローカル変換テーブルを使用: {len(id_map)} エントリ")
else:
    # フォールバック: g:Profiler APIで変換
    gp_client = GProfiler(return_dataframe=True)
    try:
        conversion = gp_client.convert(
            organism=ORGANISM,
            query=list(all_genes),
            target_namespace="HGNC"
        )
        id_map = dict(zip(conversion["incoming"], conversion["converted"]))
        print(f"g:Profiler変換完了: {len(id_map)} エントリ")
    except Exception as e:
        print(f"g:Profiler変換失敗: {e}")
        id_map = {}

# 各比較結果のsymbol列を更新
for df in all_results.values():
    if "gene_name" in df.columns:
        df["symbol"] = df["gene_name"]
    else:
        df["symbol"] = df["gene_id_stripped"].map(id_map).fillna(df["gene_id_stripped"])

# 確認
sample_df = next(iter(all_results.values()))
print("変換サンプル (最初の5件):")
print(sample_df[["gene_id", "gene_id_stripped", "symbol"]].head())


## 1. Gene Ontology / パスウェイ エンリッチメント解析 (g:Profiler)

Up/Down-regulated遺伝子を別々に解析し、GO (BP/MF/CC)、Reactome、WikiPathways のエンリッチメントを検出します。

In [ ]:
%%time
enrichment_results = {}

for comp_name, df in all_results.items():
    for direction, lfc_filter in [("Up", df["log2FoldChange"] > LFC_THRESHOLD),
                                   ("Down", df["log2FoldChange"] < -LFC_THRESHOLD)]:
        sig = df[(df["padj"] < PADJ_THRESHOLD) & lfc_filter]
        gene_list = sig["gene_id_stripped"].dropna().tolist()
        
        key = f"{comp_name}_{direction}"
        
        if len(gene_list) < 5:
            print(f"  {key}: {len(gene_list)} genes (5未満のためスキップ)")
            continue
        
        print(f"  {key}: {len(gene_list)} genes を解析中...")
        
        try:
            result = gp_client.profile(
                organism=ORGANISM,
                query=gene_list,
                sources=["GO:BP", "GO:MF", "GO:CC", "REAC", "WP"],
                user_threshold=GO_PVAL_THRESHOLD,
                significance_threshold_method="fdr",
                no_evidences=False
            )
            
            # term sizeでフィルタ
            result = result[
                (result["term_size"] >= MIN_TERM_SIZE) &
                (result["term_size"] <= MAX_TERM_SIZE)
            ].copy()
            
            result["direction"] = direction
            result["comp_name"] = comp_name
            
            enrichment_results[key] = result
            result.to_csv(os.path.join(OUTPUT_DIR, f"go_{key}.csv"), index=False)
            print(f"    → {len(result)} significant terms")
            
        except Exception as e:
            print(f"    警告: {key} の解析に失敗しました: {e}")

print(f"\n解析完了: {len(enrichment_results)} 条件の結果を取得")

## 2. GOエンリッチメント ドットプロット（ドロップダウン切替）

各比較条件のGO/パスウェイエンリッチメント結果をドットプロットで表示します。
- **X軸**: -log10(p-value)
- **ドットサイズ**: Gene Ratio (ヒットした遺伝子の割合)
- **色**: GO/パスウェイソース
- ドロップダウンで比較条件×方向を切替

In [ ]:
if not enrichment_results:
    print("エンリッチメント結果が0件のため、ドットプロットをスキップします。")
    fig_dotplot = None
else:
    source_colors = {
        "GO:BP": "#2ECC71", "GO:MF": "#9B59B6", "GO:CC": "#F39C12",
        "REAC": "#E74C3C", "WP": "#3498DB"
    }
    
    fig_dotplot = go.Figure()
    keys = list(enrichment_results.keys())
    
    for i, (key, res) in enumerate(enrichment_results.items()):
        if len(res) == 0:
            continue
        
        top = res.nsmallest(TOP_N_TERMS, "p_value").copy()
        top["-log10p"] = -np.log10(top["p_value"].clip(lower=1e-300))
        top["gene_ratio"] = top["intersection_size"] / top["query_size"]
        
        for source in top["source"].unique():
            subset = top[top["source"] == source]
            fig_dotplot.add_trace(go.Scatter(
                x=subset["-log10p"],
                y=subset["name"],
                mode="markers",
                marker=dict(
                    size=subset["gene_ratio"] * 80 + 5,
                    color=source_colors.get(source, "#95A5A6"),
                    opacity=0.8,
                    line=dict(width=1, color="white")
                ),
                name=source,
                text=[f"Term: {n}<br>Source: {s}<br>p-value: {p:.2e}<br>"
                      f"Gene Ratio: {gr:.2%}<br>Genes: {isz}/{qsz}"
                      for n, s, p, gr, isz, qsz in zip(
                          subset["name"], subset["source"], subset["p_value"],
                          subset["gene_ratio"], subset["intersection_size"],
                          subset["query_size"])],
                hovertemplate="%{text}<extra></extra>",
                visible=(i == 0),
                legendgroup=source,
                showlegend=(i == 0)
            ))
    
    # ドロップダウン作成
    # 各キーのトレース数をカウント
    traces_per_key = []
    for key, res in enrichment_results.items():
        if len(res) == 0:
            traces_per_key.append(0)
        else:
            top = res.nsmallest(TOP_N_TERMS, "p_value")
            traces_per_key.append(top["source"].nunique())
    
    total_traces = sum(traces_per_key)
    buttons = []
    for i, key in enumerate(keys):
        vis = [False] * total_traces
        offset = sum(traces_per_key[:i])
        for j in range(traces_per_key[i]):
            vis[offset + j] = True
        direction_label = "↑ Up" if key.endswith("_Up") else "↓ Down"
        comp = key.rsplit("_", 1)[0]
        buttons.append(dict(
            label=f"{comp} ({direction_label})",
            method="update",
            args=[{"visible": vis}, {"title": f"GO/Pathway Enrichment: {comp} ({direction_label})"}]
        ))
    
    first_label = buttons[0]["label"] if buttons else ""
    fig_dotplot.update_layout(
        updatemenus=[dict(type="dropdown", x=0.0, y=1.15, showactive=True, buttons=buttons)],
        title=f"GO/Pathway Enrichment: {first_label}",
        xaxis_title="-log10(p-value)",
        yaxis_title="",
        template="plotly_white",
        width=1000, height=700,
        yaxis=dict(autorange="reversed"),
        legend=dict(title="Source")
    )
    
    fig_dotplot.show()

## 3. Up vs Down エンリッチメント比較バープロット

各比較条件について、Up-regulatedとDown-regulatedの上位GO:BP termを
左右対称のバープロットで表示します。

In [ ]:
comp_names = list(all_results.keys())

# Up/Down ペアが揃っている比較条件を取得
paired_comps = []
for comp in comp_names:
    up_key = f"{comp}_Up"
    down_key = f"{comp}_Down"
    has_up = up_key in enrichment_results and len(enrichment_results[up_key]) > 0
    has_down = down_key in enrichment_results and len(enrichment_results[down_key]) > 0
    if has_up or has_down:
        paired_comps.append(comp)

if not paired_comps:
    print("Up/Downのエンリッチメント結果がないため、バープロットをスキップします。")
    fig_bar_enrich = None
else:
    fig_bar_enrich = go.Figure()
    
    for i, comp in enumerate(paired_comps):
        # Up terms (GO:BP のみ)
        up_key = f"{comp}_Up"
        if up_key in enrichment_results:
            up_bp = enrichment_results[up_key]
            up_bp = up_bp[up_bp["source"] == "GO:BP"].nsmallest(10, "p_value")
        else:
            up_bp = pd.DataFrame()
        
        # Down terms (GO:BP のみ)
        down_key = f"{comp}_Down"
        if down_key in enrichment_results:
            down_bp = enrichment_results[down_key]
            down_bp = down_bp[down_bp["source"] == "GO:BP"].nsmallest(10, "p_value")
        else:
            down_bp = pd.DataFrame()
        
        # Up bars (正の値)
        if len(up_bp) > 0:
            fig_bar_enrich.add_trace(go.Bar(
                x=-np.log10(up_bp["p_value"].clip(lower=1e-300)),
                y=up_bp["name"],
                orientation="h",
                marker_color=COLOR_UP,
                name="Up-regulated",
                hovertemplate="<b>%{y}</b><br>-log10(p): %{x:.1f}<extra>Up</extra>",
                visible=(i == 0),
                legendgroup="Up",
                showlegend=(i == 0)
            ))
        
        # Down bars (負の値で表示)
        if len(down_bp) > 0:
            fig_bar_enrich.add_trace(go.Bar(
                x=np.log10(down_bp["p_value"].clip(lower=1e-300)),  # 負の値
                y=down_bp["name"],
                orientation="h",
                marker_color=COLOR_DOWN,
                name="Down-regulated",
                hovertemplate="<b>%{y}</b><br>-log10(p): %{text}<extra>Down</extra>",
                text=[f"{v:.1f}" for v in -np.log10(down_bp["p_value"].clip(lower=1e-300))],
                visible=(i == 0),
                legendgroup="Down",
                showlegend=(i == 0)
            ))
    
    # ドロップダウン
    traces_per_comp = []
    for comp in paired_comps:
        n = 0
        if f"{comp}_Up" in enrichment_results and len(enrichment_results[f"{comp}_Up"][enrichment_results[f"{comp}_Up"]["source"] == "GO:BP"]) > 0:
            n += 1
        if f"{comp}_Down" in enrichment_results and len(enrichment_results[f"{comp}_Down"][enrichment_results[f"{comp}_Down"]["source"] == "GO:BP"]) > 0:
            n += 1
        traces_per_comp.append(n)
    
    total = sum(traces_per_comp)
    buttons = []
    for i, comp in enumerate(paired_comps):
        vis = [False] * total
        offset = sum(traces_per_comp[:i])
        for j in range(traces_per_comp[i]):
            vis[offset + j] = True
        buttons.append(dict(
            label=comp, method="update",
            args=[{"visible": vis}, {"title": f"GO:BP Enrichment (Up vs Down): {comp}"}]
        ))
    
    fig_bar_enrich.update_layout(
        updatemenus=[dict(type="dropdown", x=0.0, y=1.15, showactive=True, buttons=buttons)],
        title=f"GO:BP Enrichment (Up vs Down): {paired_comps[0]}",
        xaxis_title="-log10(p-value) ← Down | Up →",
        template="plotly_white",
        width=1000, height=600,
        barmode="relative"
    )
    fig_bar_enrich.show()

## 4. Pre-ranked GSEA

全遺伝子をlog2FoldChangeの符号付きランキングスコアで順位付けし、
Gene Set Enrichment Analysis (GSEA) を実行します。
遺伝子セットは Enrichr の GO_Biological_Process_2023 (フリーライセンス) を使用します。

In [ ]:
%%time
gsea_results = {}

for comp_name, df in all_results.items():
    print(f"GSEA実行中: {comp_name}")
    
    # ランキングメトリクス: sign(log2FC) * -log10(padj)
    df_rank = df[["gene_id_stripped", "symbol", "log2FoldChange", "padj"]].dropna(
        subset=["log2FoldChange", "padj"]
    ).copy()
    df_rank["rank_metric"] = (
        -np.log10(df_rank["padj"].clip(lower=1e-300)) *
        np.sign(df_rank["log2FoldChange"])
    )
    
    # Gene Symbol を使用 (Enrichr gene sets はシンボルベース)
    df_rank = df_rank.dropna(subset=["symbol"])
    df_rank = df_rank.drop_duplicates(subset="symbol")
    df_rank = df_rank.sort_values("rank_metric", ascending=False)
    rnk = df_rank.set_index("symbol")["rank_metric"]
    
    if len(rnk) < 100:
        print(f"  遺伝子数が少なすぎます ({len(rnk)})。スキップします。")
        continue
    
    try:
        pre_res = gp.prerank(
            rnk=rnk,
            gene_sets="GO_Biological_Process_2023",
            organism="human",
            min_size=GSEA_MIN_SIZE,
            max_size=GSEA_MAX_SIZE,
            permutation_num=GSEA_PERMUTATIONS,
            seed=42,
            no_plot=True
        )
        
        res_df = pre_res.res2d.copy()
        res_df["comparison"] = comp_name
        gsea_results[comp_name] = res_df
        res_df.to_csv(os.path.join(OUTPUT_DIR, f"gsea_{comp_name}.csv"), index=False)
        
        n_pos = (res_df["NES"] > 0).sum()
        n_neg = (res_df["NES"] < 0).sum()
        sig = res_df[res_df["FDR q-val"] < 0.25]
        print(f"  Gene sets: {len(res_df)} (正: {n_pos}, 負: {n_neg}, FDR<0.25: {len(sig)})")
        
    except Exception as e:
        print(f"  警告: GSEA失敗 - {e}")

print(f"\nGSEA完了: {len(gsea_results)} 条件")

## 5. GSEA NESバープロット

Normalized Enrichment Score (NES) でソートした上位のGene Setを表示します。
- **赤**: 正のNES (Up方向にエンリッチ)
- **青**: 負のNES (Down方向にエンリッチ)

In [ ]:
if not gsea_results:
    print("GSEA結果がないため、NESバープロットをスキップします。")
    fig_gsea = None
else:
    fig_gsea = go.Figure()
    gsea_keys = list(gsea_results.keys())
    
    for i, (comp_name, res) in enumerate(gsea_results.items()):
        # FDR < 0.25 のみ、NES上位/下位各10
        sig = res[res["FDR q-val"] < 0.25].copy()
        if len(sig) == 0:
            sig = res.copy()  # FDR閾値以下がなければ全体から
        
        # NES上位10 + 下位10
        top_pos = sig.nlargest(10, "NES")
        top_neg = sig.nsmallest(10, "NES")
        top = pd.concat([top_pos, top_neg]).drop_duplicates(subset="Term")
        top = top.sort_values("NES")
        
        # Term名を短縮 (Enrichr形式: "term (GO:XXXXXXX)" → "term")
        top["term_short"] = top["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)
        # 長すぎるterm名を切り詰め
        top["term_short"] = top["term_short"].apply(lambda x: x[:60] + "..." if len(str(x)) > 60 else x)
        
        colors = [COLOR_UP if nes > 0 else COLOR_DOWN for nes in top["NES"]]
        
        fig_gsea.add_trace(go.Bar(
            x=top["NES"],
            y=top["term_short"],
            orientation="h",
            marker_color=colors,
            text=[f"NES: {nes:.2f}<br>FDR: {fdr:.3f}<br>Size: {sz}"
                  for nes, fdr, sz in zip(top["NES"], top["FDR q-val"], top["Tag %"])],
            hovertemplate="<b>%{y}</b><br>%{text}<extra></extra>",
            visible=(i == 0),
            showlegend=False
        ))
    
    buttons = []
    for i, comp_name in enumerate(gsea_keys):
        vis = [False] * len(gsea_keys)
        vis[i] = True
        buttons.append(dict(
            label=comp_name, method="update",
            args=[{"visible": vis}, {"title": f"GSEA NES: {comp_name}"}]
        ))
    
    fig_gsea.update_layout(
        updatemenus=[dict(type="dropdown", x=0.0, y=1.15, showactive=True, buttons=buttons)],
        title=f"GSEA NES: {gsea_keys[0]}",
        xaxis_title="Normalized Enrichment Score (NES)",
        template="plotly_white",
        width=1000, height=700
    )
    fig_gsea.add_vline(x=0, line_color="grey", line_dash="dash", opacity=0.5)
    fig_gsea.show()

## 6. エンリッチメントネットワーク (Cytoscape風)

エンリッチされたGO/パスウェイ term をノードとし、共通遺伝子の重なり（Jaccard係数）を
エッジとするネットワークを構築します。Cytoscape の EnrichmentMap に相当する可視化です。

- **ノードサイズ**: ヒット遺伝子数
- **ノード色**: -log10(p-value)
- **エッジ太さ**: Jaccard係数（遺伝子の共有度）

In [ ]:
def build_term_network(enrich_df, top_n=TOP_N_TERMS, threshold=JACCARD_THRESHOLD):
    """エンリッチされたtermのネットワークを構築"""
    G = nx.Graph()
    top = enrich_df.nsmallest(top_n, "p_value").copy()
    
    if len(top) < 2:
        return G, {}
    
    # ノード追加
    term_genes = {}
    for _, row in top.iterrows():
        name = row["name"]
        genes = set(row["intersections"]) if isinstance(row["intersections"], list) else set()
        term_genes[name] = genes
        G.add_node(name,
                   p_value=row["p_value"],
                   size=row["intersection_size"],
                   source=row["source"],
                   term_id=row.get("native", ""))
    
    # Jaccard係数に基づくエッジ追加
    terms = list(term_genes.keys())
    for i in range(len(terms)):
        for j in range(i + 1, len(terms)):
            t1, t2 = terms[i], terms[j]
            union = term_genes[t1] | term_genes[t2]
            if len(union) == 0:
                continue
            jaccard = len(term_genes[t1] & term_genes[t2]) / len(union)
            if jaccard >= threshold:
                G.add_edge(t1, t2, weight=jaccard)
    
    return G, term_genes


def plot_network(G, title="Enrichment Network"):
    """networkxグラフをplotlyでインタラクティブに描画"""
    if len(G.nodes) == 0:
        print(f"ノードが0件のため {title} をスキップします。")
        return None
    
    # レイアウト計算
    pos = nx.spring_layout(G, k=2.0, iterations=50, seed=42)
    
    # エッジトレース
    edge_traces = []
    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        weight = data.get("weight", 0.3)
        edge_traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode="lines",
            line=dict(width=weight * 8, color="#CCCCCC"),
            hoverinfo="none",
            showlegend=False
        ))
    
    # ノードデータ
    node_x, node_y, node_size, node_color, node_text, node_hover = [], [], [], [], [], []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        
        data = G.nodes[node]
        pval = data.get("p_value", 1.0)
        size = data.get("size", 1)
        source = data.get("source", "")
        
        node_size.append(max(15, min(60, size * 3)))
        node_color.append(-np.log10(max(pval, 1e-300)))
        
        # ラベル (短縮)
        label = node[:35] + "..." if len(node) > 35 else node
        node_text.append(label)
        node_hover.append(
            f"<b>{node}</b><br>"
            f"Source: {source}<br>"
            f"p-value: {pval:.2e}<br>"
            f"Gene count: {size}<br>"
            f"Connections: {G.degree(node)}"
        )
    
    # ノードトレース
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode="markers+text",
        marker=dict(
            size=node_size,
            color=node_color,
            colorscale="YlOrRd",
            colorbar=dict(title="-log10(p)", len=0.5),
            line=dict(width=1.5, color="white")
        ),
        text=node_text,
        textposition="top center",
        textfont=dict(size=9),
        hovertemplate="%{customdata}<extra></extra>",
        customdata=node_hover,
        showlegend=False
    )
    
    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1000, height=800,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        hovermode="closest"
    )
    return fig

print("ネットワーク構築関数を定義しました")

In [ ]:
# 各比較×方向でネットワークを構築・表示
network_figs = {}

for key, res in enrichment_results.items():
    if len(res) < 2:
        print(f"{key}: termが2未満のためスキップ")
        continue
    
    G, term_genes = build_term_network(res, top_n=TOP_N_TERMS, threshold=JACCARD_THRESHOLD)
    
    if len(G.nodes) == 0:
        print(f"{key}: ネットワークノードが0件")
        continue
    
    direction_label = "Up ↑" if key.endswith("_Up") else "Down ↓"
    comp = key.rsplit("_", 1)[0]
    title = f"Enrichment Network: {comp} ({direction_label})"
    
    fig_net = plot_network(G, title=title)
    if fig_net is not None:
        network_figs[key] = fig_net
        fig_net.show()
        print(f"  {key}: {len(G.nodes)} nodes, {len(G.edges)} edges")

if not network_figs:
    print("ネットワーク可視化可能な条件がありませんでした。")

## 7. 全条件エンリッチメント比較ヒートマップ

全ての比較条件にわたるGO:BP termの-log10(p-value)をヒートマップで一覧表示します。
条件間で共通・固有のパスウェイ変動を把握できます。

In [ ]:
if not enrichment_results:
    print("エンリッチメント結果がないため、ヒートマップをスキップします。")
    fig_enrich_heat = None
else:
    # 全条件のGO:BP結果を統合し、上位termを選出
    all_bp = []
    for key, res in enrichment_results.items():
        bp = res[res["source"] == "GO:BP"].copy()
        if len(bp) > 0:
            bp["key"] = key
            all_bp.append(bp)
    
    if not all_bp:
        print("GO:BP結果がないため、ヒートマップをスキップします。")
        fig_enrich_heat = None
    else:
        all_bp_df = pd.concat(all_bp)
        
        # 最も頻出 or 最も有意なterm上位30を選出
        top_terms = (all_bp_df.groupby("name")["p_value"]
                     .min()
                     .nsmallest(30)
                     .index.tolist())
        
        # -log10(p) 行列を構築
        keys = list(enrichment_results.keys())
        heatmap_data = pd.DataFrame(0.0, index=top_terms, columns=keys)
        
        for key, res in enrichment_results.items():
            bp = res[res["source"] == "GO:BP"].set_index("name")
            for term in top_terms:
                if term in bp.index:
                    heatmap_data.loc[term, key] = -np.log10(
                        max(bp.loc[term, "p_value"] if isinstance(bp.loc[term, "p_value"], float) 
                            else bp.loc[term, "p_value"].iloc[0], 1e-300)
                    )
        
        # クラスタリング
        from scipy.cluster.hierarchy import linkage, leaves_list
        from scipy.spatial.distance import pdist
        
        if len(heatmap_data) > 1:
            dist = pdist(heatmap_data.values, metric="euclidean")
            link = linkage(dist, method="ward")
            order = leaves_list(link)
            heatmap_data = heatmap_data.iloc[order]
        
        # ホバーテキスト
        hover = []
        for term in heatmap_data.index:
            row = []
            for key in heatmap_data.columns:
                val = heatmap_data.loc[term, key]
                row.append(f"Term: {term}<br>Condition: {key}<br>-log10(p): {val:.1f}")
            hover.append(row)
        
        # カラム名を短縮表示
        col_labels = []
        for k in heatmap_data.columns:
            d = "Up" if k.endswith("_Up") else "Down"
            c = k.rsplit("_", 1)[0]
            col_labels.append(f"{c}\n({d})")
        
        fig_enrich_heat = go.Figure(data=go.Heatmap(
            z=heatmap_data.values,
            x=col_labels,
            y=heatmap_data.index.tolist(),
            colorscale="YlOrRd",
            text=hover, hoverinfo="text",
            colorbar=dict(title="-log10(p)")
        ))
        
        fig_h = max(600, min(1500, len(heatmap_data) * 25 + 200))
        fig_enrich_heat.update_layout(
            title="全条件 GO:BP エンリッチメント比較ヒートマップ",
            template="plotly_white",
            width=max(700, len(heatmap_data.columns) * 150 + 400),
            height=fig_h,
            yaxis=dict(tickfont=dict(size=9))
        )
        fig_enrich_heat.show()

## 8. 結果エクスポート

In [ ]:
# HTML出力
saved_files = []

if fig_dotplot is not None:
    path = os.path.join(OUTPUT_DIR, "go_dotplot_interactive.html")
    fig_dotplot.write_html(path, include_plotlyjs=True)
    saved_files.append(path)

if fig_bar_enrich is not None:
    path = os.path.join(OUTPUT_DIR, "go_barplot_updown_interactive.html")
    fig_bar_enrich.write_html(path, include_plotlyjs=True)
    saved_files.append(path)

if fig_gsea is not None:
    path = os.path.join(OUTPUT_DIR, "gsea_nes_barplot_interactive.html")
    fig_gsea.write_html(path, include_plotlyjs=True)
    saved_files.append(path)

for key, fig_net in network_figs.items():
    path = os.path.join(OUTPUT_DIR, f"network_{key}_interactive.html")
    fig_net.write_html(path, include_plotlyjs=True)
    saved_files.append(path)

if fig_enrich_heat is not None:
    path = os.path.join(OUTPUT_DIR, "enrichment_heatmap_interactive.html")
    fig_enrich_heat.write_html(path, include_plotlyjs=True)
    saved_files.append(path)

print("=== 出力ファイル一覧 ===")
for f in saved_files:
    print(f"  {f}")

# CSV一覧
csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))
print(f"\n=== CSV結果ファイル ({len(csv_files)}件) ===")
for f in csv_files:
    print(f"  {f}")

## 完了

### 生成されたファイル

| ファイル | 内容 |
|---------|------|
| `results/functional/go_*_Up.csv` / `go_*_Down.csv` | 各比較条件のGO/パスウェイエンリッチメント結果 |
| `results/functional/gsea_*.csv` | 各比較条件のGSEA結果 |
| `results/functional/go_dotplot_interactive.html` | GOエンリッチメント ドットプロット |
| `results/functional/go_barplot_updown_interactive.html` | Up vs Down GO:BPバープロット |
| `results/functional/gsea_nes_barplot_interactive.html` | GSEA NESバープロット |
| `results/functional/network_*_interactive.html` | エンリッチメントネットワーク (Cytoscape風) |
| `results/functional/enrichment_heatmap_interactive.html` | 全条件比較エンリッチメントヒートマップ |

### 操作方法
- HTMLファイルをブラウザで開くと、インタラクティブに操作できます
- **ドットプロット / バープロット**: ドロップダウンで比較条件を切替
- **ネットワーク**: マウスホバーでterm詳細表示、ドラッグでズーム・パン
- **ヒートマップ**: マウスホバーでterm名・条件・-log10(p)を表示

### ライセンスについて
本ノートブックで使用する全てのツール・データベースは商用利用可能なライセンスです:
- **g:Profiler API** (BSD) → GO (CC BY 4.0), Reactome (CC BY 4.0), WikiPathways (CC BY 3.0)
- **gseapy** (BSD-3) → Enrichr gene sets (CC)
- **networkx** (BSD-3), **plotly** (MIT)
- KEGG は商用ライセンスが必要なため**意図的に除外**しています